In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from other_methods import delves_lyness, kravanja_van_barel
from scipy.stats import qmc
from tqdm import tqdm

from skzeros._domain import Rectangle, _subdivide_domain, get_leaf_regions
from skzeros._zeros import find_zeros

plt.style.use("style.mplstyle")

In [ ]:
N = 100
rng = np.random.default_rng(12345)

In [ ]:
sampler = qmc.Sobol(d=2, rng=rng)
tmp = sampler.random(N)
zeros = tmp[:, 0] + tmp[:, 1] * 1j

In [ ]:
count = 0


def f(z):
    global count  # noqa: PLW0603
    count += getattr(z, "size", 1)
    return np.prod(np.subtract.outer(z, zeros), axis=-1)


def f_z(z):
    tmp = np.subtract.outer(z, zeros)
    return np.prod(tmp, axis=-1) * np.sum(1 / tmp, axis=-1)

In [ ]:
count = 0
d = Rectangle(0, complex(1, 1))
res = find_zeros(
    d,
    f,
    f_z,
    max_arg_principle=20.1,
    quadrature_args={"epsabs": 1e-1, "epsrel": 1e-1},
    rng=rng,
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(4, 2))
d.plot(ax1)
ax1.plot(res.zeros.real, res.zeros.imag, ".")
ax1.set_xlabel(r"$\Re(z)$")
ax1.set_ylabel(r"$\Im(z)$")
ax1.set_aspect("equal", "box")
ax2.hist(np.log10(np.min(np.abs(np.subtract.outer(zeros, res.zeros)), axis=-1)))
ax2.set_xlim([-16, -5])
ax2.set_xlabel("Log Error")
ax2.set_ylabel("Frequency")
plt.tight_layout()
plt.savefig("figures/quasi_random.pdf")

In [ ]:
AAA_err = np.min(np.abs(np.subtract.outer(zeros, res.zeros)), axis=-1)
AAA_count = count
print(
    f"AAA: {count=}, max error={np.max(AAA_err):.1e}, min error={np.min(AAA_err):.1e}, medium error={np.median(AAA_err):.1e}"
)
print(len(get_leaf_regions(d)))

In [ ]:
count = 0
delves_lyness_zeros = []
d = Rectangle(0, complex(1, 1))
regions = _subdivide_domain(
    d,
    f,
    f_z,
    max_arg_principle=5.1,
    maxiter=100,
    quadrature_args={"epsabs": 1e-1, "epsrel": 1e-1},
    rng=rng,
)
for region in tqdm(regions):
    # Get the expected number of zeros
    N = region._arg_principle
    # if there are no zeros then there is no work to be done!
    if np.isclose(N, 0, atol=1e-3):
        continue
    delves_lyness_zeros.append(
        delves_lyness(
            f,
            f_z,
            region,
            N=np.round(N.real).astype(int),
            quad_kwargs={"epsabs": 1e-12, "epsrel": 1e-12},
        )
    )

delves_lyness_zeros = np.concatenate(delves_lyness_zeros)

delves_lyness_err = np.min(
    np.abs(np.subtract.outer(zeros, delves_lyness_zeros)), axis=-1
)
delves_lyness_count = count
print(
    f"Delves-Lyness: {count=}, max error={np.max(delves_lyness_err):.1e}, min error={np.min(delves_lyness_err):.1e}, medium error={np.median(delves_lyness_err):.1e}"
)
print(len(get_leaf_regions(d)))

In [ ]:
count = 0
kvb_zeros = []
d = Rectangle(0, complex(1, 1))
regions = _subdivide_domain(
    d,
    f,
    f_z,
    max_arg_principle=2.1,
    maxiter=200,
    quadrature_args={"epsabs": 1e-1, "epsrel": 1e-1},
    rng=rng,
)
for region in tqdm(regions):
    # Get the expected number of zeros
    N = region._arg_principle
    # if there are no zeros then there is no work to be done!
    if np.isclose(N, 0, atol=1e-3):
        continue
    kvb_zeros.append(
        kravanja_van_barel(
            f,
            f_z,
            region,
            N=np.round(N.real).astype(int),
            quad_kwargs={"epsabs": 1e-12, "epsrel": 1e-12},
        )[0]
    )

kvb_zeros = np.concatenate(kvb_zeros)

kvb_err = np.min(np.abs(np.subtract.outer(zeros, kvb_zeros)), axis=0)
kvb_count = count
print(
    f"kvb: {count=}, max error={np.max(kvb_err):.1e}, min error={np.min(kvb_err):.1e}, medium error={np.median(kvb_err):.1e}"
)
print(len(get_leaf_regions(d)))

In [ ]:
count = 0
from cxroots import Rectangle as cxRectangle
from cxroots.root_approximation import approximate_roots

kvb_fop_zeros = []
d = Rectangle(0, complex(1, 1))
regions = _subdivide_domain(
    d,
    f,
    f_z,
    max_arg_principle=5.1,
    maxiter=200,
    quadrature_args={"epsabs": 1e-1, "epsrel": 1e-1},
    rng=rng,
)
for region in tqdm(regions):
    # Get the expected number of zeros
    N = region._arg_principle
    # if there are no zeros then there is no work to be done!
    if np.isclose(N, 0, atol=1e-3):
        continue
    c = cxRectangle(
        (region.bottom_left.real, region.top_right.real),
        (region.bottom_left.imag, region.top_right.imag),
    )
    result = approximate_roots(
        c, np.round(N.real).astype(int), f, f_z, abs_tol=1e-12, rel_tol=1e-12
    )
    kvb_fop_zeros.append(result[0])

kvb_fop_zeros = np.concatenate(kvb_fop_zeros)
kvb_err = np.min(np.abs(np.subtract.outer(zeros, kvb_fop_zeros)), axis=0)
kvb_count = count
print(
    f"kvb_fop: {count=}, max error={np.max(kvb_err):.1e}, min error={np.min(kvb_err):.1e}, medium error={np.median(kvb_err):.1e}"
)
print(len(get_leaf_regions(d)))